In [12]:
! pip install --upgrade pip
!pip install kagglehub
!pip install Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 9.7 MB/s  0:00:009.3 MB/s eta 0:00:01


In [13]:
import kagglehub
import os
from PIL import Image
import random

In [5]:

# Download latest version
path = kagglehub.dataset_download("minhtmnguyntrn/affectnet-dataset")

print("Path to dataset files:", path)

/Users/bibo/dev/vevns/EmotionDetector/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.74G/1.74G [01:16<00:00, 24.4MB/s]

Extracting files...


Path to dataset files: /Users/bibo/.cache/kagglehub/datasets/minhtmnguyntrn/affectnet-dataset/versions/1


In [25]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
data_path = os.path.join(path, "data")  # path from kagglehub

TARGET_SIZE = 160
IMAGES_PER_CLASS = 800
DROP_CLASS = '7'  # Contempt

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

label_map = {
    '0': 'neutral',
    '1': 'happy',
    '2': 'sad',
    '3': 'surprise',
    '4': 'fear',
    '5': 'disgust',
    '6': 'angry'
}

output_path = os.path.join(NOTEBOOK_DIR, "emotion_dataset")
os.makedirs(output_path, exist_ok=True)
print("Output path:", output_path)

Output path: /Users/bibo/dev/MSc/HardSofCod/EmotionDetector/emotion_dataset


In [26]:
valid_images_per_class = {}

for class_folder in sorted(os.listdir(data_path)):
    if class_folder == DROP_CLASS:
        print(f"Skipping class {class_folder} (Contempt)")
        continue

    class_path = os.path.join(data_path, class_folder)
    if not os.path.isdir(class_path):
        continue

    valid = []
    for img_file in os.listdir(class_path):
        img_full_path = os.path.join(class_path, img_file)
        try:
            with Image.open(img_full_path) as img:
                w, h = img.size
                if w >= TARGET_SIZE and h >= TARGET_SIZE:
                    valid.append(img_file)
        except Exception:
            pass

    valid_images_per_class[class_folder] = valid
    print(f"Class {class_folder} ({label_map[class_folder]}): {len(valid)} valid images")

Class 0 (neutral): 5418 valid images
Class 1 (happy): 4221 valid images
Class 2 (sad): 5389 valid images
Class 3 (surprise): 5434 valid images
Class 4 (fear): 5423 valid images
Class 5 (disgust): 5431 valid images
Class 6 (angry): 5414 valid images
Skipping class 7 (Contempt)


In [27]:
selected_per_class = {}

for class_folder, valid in valid_images_per_class.items():
    if len(valid) < IMAGES_PER_CLASS:
        print(f"Class {class_folder} ({label_map[class_folder]}): only {len(valid)} available, taking all")
        selected_per_class[class_folder] = valid
    else:
        selected_per_class[class_folder] = random.sample(valid, IMAGES_PER_CLASS)
        print(f"Class {class_folder} ({label_map[class_folder]}): sampled {IMAGES_PER_CLASS} images")

Class 0 (neutral): sampled 800 images
Class 1 (happy): sampled 800 images
Class 2 (sad): sampled 800 images
Class 3 (surprise): sampled 800 images
Class 4 (fear): sampled 800 images
Class 5 (disgust): sampled 800 images
Class 6 (angry): sampled 800 images


In [28]:
splits_per_class = {}

for class_folder, selected in selected_per_class.items():
    random.shuffle(selected)
    
    n = len(selected)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    # test gets the remainder to ensure no images are lost
    
    splits_per_class[class_folder] = {
        'train': selected[:n_train],
        'val':   selected[n_train:n_train + n_val],
        'test':  selected[n_train + n_val:]
    }
    
    print(f"Class {label_map[class_folder]}: "
          f"train={len(splits_per_class[class_folder]['train'])} "
          f"val={len(splits_per_class[class_folder]['val'])} "
          f"test={len(splits_per_class[class_folder]['test'])}")

Class neutral: train=560 val=120 test=120
Class happy: train=560 val=120 test=120
Class sad: train=560 val=120 test=120
Class surprise: train=560 val=120 test=120
Class fear: train=560 val=120 test=120
Class disgust: train=560 val=120 test=120
Class angry: train=560 val=120 test=120


In [29]:
for class_folder, splits in splits_per_class.items():
    emotion_name = label_map[class_folder]
    class_path = os.path.join(data_path, class_folder)

    for split_name, files in splits.items():
        out_dir = os.path.join(output_path, split_name, emotion_name)
        os.makedirs(out_dir, exist_ok=True)

        saved = 0
        for img_file in files:
            img_full_path = os.path.join(class_path, img_file)
            out_img_path  = os.path.join(out_dir, img_file)
            try:
                with Image.open(img_full_path) as img:
                    img = img.convert('RGB')
                    img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
                    img.save(out_img_path, 'JPEG', quality=95)
                    saved += 1
            except Exception as e:
                print(f"Error on {img_file}: {e}")

        print(f" {split_name}/{emotion_name}: saved {saved} images")

 train/neutral: saved 560 images
 val/neutral: saved 120 images
 test/neutral: saved 120 images
 train/happy: saved 560 images
 val/happy: saved 120 images
 test/happy: saved 120 images
 train/sad: saved 560 images
 val/sad: saved 120 images
 test/sad: saved 120 images
 train/surprise: saved 560 images
 val/surprise: saved 120 images
 test/surprise: saved 120 images
 train/fear: saved 560 images
 val/fear: saved 120 images
 test/fear: saved 120 images
 train/disgust: saved 560 images
 val/disgust: saved 120 images
 test/disgust: saved 120 images
 train/angry: saved 560 images
 val/angry: saved 120 images
 test/angry: saved 120 images


In [30]:
print("Final dataset summary:")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(output_path, split)
    print(f"\n  {split}/")
    split_total = 0
    for emotion in sorted(os.listdir(split_path)):
        emotion_path = os.path.join(split_path, emotion)
        if os.path.isdir(emotion_path):
            count = len(os.listdir(emotion_path))
            split_total += count
            print(f"    {emotion}: {count} images")
    print(f"    subtotal: {split_total}")

Final dataset summary:

  train/
    angry: 560 images
    disgust: 560 images
    fear: 560 images
    happy: 560 images
    neutral: 560 images
    sad: 560 images
    surprise: 560 images
    subtotal: 3920

  val/
    angry: 120 images
    disgust: 120 images
    fear: 120 images
    happy: 120 images
    neutral: 120 images
    sad: 120 images
    surprise: 120 images
    subtotal: 840

  test/
    angry: 120 images
    disgust: 120 images
    fear: 120 images
    happy: 120 images
    neutral: 120 images
    sad: 120 images
    surprise: 120 images
    subtotal: 840
